In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (train_test_split, GridSearchCV)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, roc_auc_score, classification_report, ConfusionMatrixDisplay)
SEED = 100


In [0]:
transactions = spark.table("fraud_transactions_for_detection").toPandas()

# Stable ordering makes the random split reproducible.
transactions = (transactions.sort_values("Transaction_ID").reset_index(drop=True))


print("Cleaned transactions:", len(transactions))

In [0]:
development, test = train_test_split(
    transactions,
    test_size=0.20,
    stratify=transactions["Fraudulent"],
    random_state=SEED
)

train, validation = train_test_split(
    development,
    test_size=0.25,
    stratify=development["Fraudulent"],
    random_state=SEED
)

assert len(train) + len(validation) + len(test) == len(transactions)

display(pd.DataFrame([
    {
        "set": name,
        "transactions": len(frame),
        "fraud_cases": int(frame["Fraudulent"].sum()),
        "fraud_rate_pct": 100 * frame["Fraudulent"].mean()
    }
    for name, frame in [("Training", train), ("Validation", validation),("Test", test)]
]))

In [0]:
numeric_features = [
    "Transaction_Amount", "Account_Age", "Number_of_Transactions_Last_24H", "hour_sin","hour_cos"]

categorical_features = [
    "Transaction_Type","Device_Used","Location","Payment_Method"]

def make_features(frame):
    features = frame.copy()

    hour = pd.to_numeric(
        features["Time_of_Transaction"],
        errors="raise"
    )
    assert hour.dropna().between(0, 23).all()

    features["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    features["hour_cos"] = np.cos(2 * np.pi * hour / 24)

    for column in numeric_features:
        features[column] = pd.to_numeric(
            features[column], errors="raise"
        )
        assert not np.isinf(
            features[column].to_numpy(dtype=float)
        ).any()

    for column in categorical_features:
        features[column] = (
            features[column]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

    return features[numeric_features + categorical_features]

X_train = make_features(train)
X_val = make_features(validation)

y_train = train["Fraudulent"].astype(int)
y_val = validation["Fraudulent"].astype(int)

print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)

In [0]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median",add_indicator=True)),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore",sparse_output=False), 
     categorical_features)
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(class_weight="balanced", solver="lbfgs", max_iter=2000, random_state=SEED))
])

In [0]:

search = GridSearchCV(
    pipeline, param_grid={"classifier__C": [0.1, 1.0, 10.0]}, cv=5, scoring="average_precision")

search.fit(X_train, y_train)

best_model = search.best_estimator_

print("Best parameters:", search.best_params_)
print("Cross-validation average precision:", search.best_score_)

In [0]:
fraud_class_index = int(
    np.flatnonzero(best_model.classes_ == 1)[0]
)

val_scores = best_model.predict_proba(
    X_val
)[:, fraud_class_index]

val_predictions = best_model.predict(X_val)

print("Validation average precision:",
      round(average_precision_score(y_val, val_scores), 4))

print("Validation ROC-AUC:",
      round(roc_auc_score(y_val, val_scores), 4))

print("Alert rate:",
      f"{100 * val_predictions.mean():.2f}%")

print(
    classification_report(
        y_val,
        val_predictions,
        labels=[0, 1],
        target_names=["Legitimate", "Fraudulent"],
        digits=4,
        zero_division=0
    )
)

ConfusionMatrixDisplay.from_predictions(
    y_val,
    val_predictions,
    labels=[0, 1],
    display_labels=["Legitimate", "Fraudulent"],
    cmap="Blues",
    values_format="d"
)

plt.title("Balanced logistic regression — validation")
plt.show()